In [ ]:
# CELL 1 — imports + load
import json, sys, os
sys.path.insert(0, os.path.abspath('..'))
import numpy as np
import pandas as pd
import lightgbm as lgb
import optuna
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error
from src.features import build_modeling_frame

optuna.logging.set_verbosity(optuna.logging.WARNING)
df = pd.read_parquet('../data/processed/train.parquet')
X, y, cat_cols = build_modeling_frame(df)
print('X:', X.shape, '  y:', y.shape, '  cats:', len(cat_cols))

In [ ]:
# CELL 2 — Optuna objective (3-fold CV inside, 5-min hard cap)
def objective(trial):
    params = {
        'n_estimators': 800,
        'learning_rate': trial.suggest_float('learning_rate', 0.02, 0.08, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 31, 127),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 60),
        'feature_fraction': trial.suggest_float('feature_fraction', 0.7, 1.0),
        'bagging_fraction': trial.suggest_float('bagging_fraction', 0.7, 1.0),
        'bagging_freq': 5,
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 1.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 1.0, log=True),
        'random_state': 42, 'n_jobs': -1, 'verbose': -1,
    }
    kf = KFold(n_splits=3, shuffle=True, random_state=42)
    maes = []
    for tr, va in kf.split(X):
        m = lgb.LGBMRegressor(**params)
        m.fit(X.iloc[tr], y.iloc[tr], categorical_feature=cat_cols,
              eval_set=[(X.iloc[va], y.iloc[va])],
              callbacks=[lgb.early_stopping(40, verbose=False)])
        maes.append(mean_absolute_error(y.iloc[va], m.predict(X.iloc[va])))
    return float(np.mean(maes))

study = optuna.create_study(direction='minimize',
                            sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=60, timeout=300, show_progress_bar=True)
print('Best MAE (3-fold):', study.best_value)
print('Best params:', study.best_params)

In [ ]:
# CELL 3 — final 5-fold with best params; save model + OOF
best = dict(study.best_params)
best.update(dict(n_estimators=1200, bagging_freq=5,
                 random_state=42, n_jobs=-1, verbose=-1))
kf = KFold(n_splits=5, shuffle=True, random_state=42)
maes = []
oof_lgbm = np.zeros(len(y))
final_model = None
for fold, (tr, va) in enumerate(kf.split(X), 1):
    m = lgb.LGBMRegressor(**best)
    m.fit(X.iloc[tr], y.iloc[tr], categorical_feature=cat_cols,
          eval_set=[(X.iloc[va], y.iloc[va])],
          callbacks=[lgb.early_stopping(50, verbose=False)])
    p = m.predict(X.iloc[va])
    oof_lgbm[va] = p
    mae = mean_absolute_error(y.iloc[va], p)
    maes.append(mae)
    print(f'Fold {fold} MAE = {mae:.3f}')
    if fold == 5:
        final_model = m
print(f'\nTuned mean MAE = {np.mean(maes):.3f} ± {np.std(maes):.3f}')

In [ ]:
# CELL 4 — save tuned scores + model
with open('../reports/scores_v1_tuned.json', 'w') as f:
    json.dump({
        'fold_mae': [float(x) for x in maes],
        'mae_mean': float(np.mean(maes)),
        'mae_std':  float(np.std(maes)),
        'best_params': best,
    }, f, indent=2)
final_model.booster_.save_model('../models/lgbm_v1.txt')
np.save('../data/processed/oof_lgbm_v1.npy', oof_lgbm)
print('Saved scores + model + OOF')

In [ ]:
# CELL 5 — CatBoost 5-fold on same split
from catboost import CatBoostRegressor, Pool

Xc = X.copy()
for c in cat_cols:
    Xc[c] = Xc[c].astype('object').fillna('missing').astype(str)

cb_maes = []
oof_cb = np.zeros(len(y))
for fold, (tr, va) in enumerate(kf.split(Xc), 1):
    train_pool = Pool(Xc.iloc[tr], y.iloc[tr], cat_features=cat_cols)
    valid_pool = Pool(Xc.iloc[va], y.iloc[va], cat_features=cat_cols)
    m = CatBoostRegressor(
        iterations=1500, depth=8, learning_rate=0.05,
        loss_function='MAE', eval_metric='MAE',
        random_seed=42, early_stopping_rounds=50, verbose=False,
    )
    m.fit(train_pool, eval_set=valid_pool, use_best_model=True)
    p = m.predict(Xc.iloc[va])
    oof_cb[va] = p
    mae = mean_absolute_error(y.iloc[va], p)
    cb_maes.append(mae)
    print(f'CatBoost fold {fold} MAE = {mae:.3f}  (best_iter={m.best_iteration_})')
print(f'\nCatBoost mean MAE = {np.mean(cb_maes):.3f} ± {np.std(cb_maes):.3f}')

In [ ]:
# CELL 6 — weighted blend (0.55 LGBM + 0.45 CatBoost) on same OOF split
w_lgbm, w_cb = 0.55, 0.45
oof_blend = w_lgbm * oof_lgbm + w_cb * oof_cb

blend_fold_maes = [mean_absolute_error(y.iloc[va], oof_blend[va]) for _, va in kf.split(Xc)]

lgbm_mean = float(mean_absolute_error(y, oof_lgbm))
cb_mean = float(np.mean(cb_maes))
blend_mean = float(np.mean(blend_fold_maes))

print(f'LGBM tuned (OOF)  MAE = {lgbm_mean:.4f}')
print(f'CatBoost          MAE = {cb_mean:.4f}')
print(f'Blend 0.55/0.45   MAE = {blend_mean:.4f}')
if blend_mean < lgbm_mean:
    print(f'Blend BEATS LGBM by {lgbm_mean - blend_mean:.4f} MAE')
else:
    print(f'Blend DOES NOT beat LGBM (worse by {blend_mean - lgbm_mean:.4f} MAE)')

with open('../reports/scores_v1_blend.json', 'w') as f:
    json.dump({
        'weights': {'lgbm': w_lgbm, 'catboost': w_cb},
        'lgbm_oof_mae': lgbm_mean,
        'catboost_fold_mae': [float(x) for x in cb_maes],
        'catboost_mean_mae': cb_mean,
        'blend_fold_mae': [float(x) for x in blend_fold_maes],
        'blend_mean_mae': blend_mean,
        'blend_std_mae': float(np.std(blend_fold_maes)),
        'beats_lgbm': bool(blend_mean < lgbm_mean),
        'improvement_over_lgbm': float(lgbm_mean - blend_mean),
    }, f, indent=2)
print('Saved reports/scores_v1_blend.json')